# Middleware

- Middleware provides a way to more tightly control what happenes inside the agent. Middleware is useful for the following:

 1. Tracking agent behaviour with logging, analytics and debugging.
 2. Transforming prompts, tool selection, and output formatting.
 3. Adding retries, fallbacks, and early termination logic.
 4. Applying rate limits, guardrails, and  PII detection

# hooks means triggers
# built in middlerware 
 1. Summarization -> Agents
 2. Human in the feedback
 3. Modelcall limit

## Summarization Middleware
- Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
 1. Long-running converstaions that exceed context windows
 2. Multi- turn dialogues with extensive history.
 3. Applications where preserving full conversation context matters


In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage


# Message-based summarization
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash-lite",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)


# Run with thread ID
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4^4?",
]


for q in questions:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=q)
            ]
        },
        config,
    )

    print(f"Question: {q}")
    print(f"Response: {response['messages'][-1].content}")
    print(f"Message count: {len(response['messages'])}")
    print("-" * 50)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Question: What is 2+2?
Response: 2 + 2 = 4
Message count: 2
--------------------------------------------------
Question: What is 10*5?
Response: 10 * 5 = 50
Message count: 4
--------------------------------------------------
Question: What is 100/4?
Response: 100 / 4 = 25
Message count: 6
--------------------------------------------------
Question: What is 15-7?
Response: 15 - 7 = 8
Message count: 8
--------------------------------------------------
Question: What is 3*3?
Response: 3 * 3 = 9
Message count: 10
--------------------------------------------------
Question: What is 4^4?
Response: 4^4 = 256
Message count: 6
--------------------------------------------------


# Token size

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels and return hotel information for a city."""
    return f"""
Hotels in {city}:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $100/night, business center
3. Budget Stay - 3 star, $75/night, free wifi
"""


agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        )
    ],
)


config = {
    "configurable": {
        "thread_id": "text-1"
    }
}


# Approximate token counter
def count_tokens(messages):
    total_chars = sum(
        len(str(message.content))
        for message in messages
    )
    return total_chars // 4


# Run test
cities = [
    "Paris",
    "London",
    "Tokyo",
    "New York",
    "Dubai",
    "Singapore",
]


for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Find hotels in {city}"
                )
            ]
        },
        config=config,
    )

    messages = response["messages"]
    tokens = count_tokens(messages)

    print(f"\n{'=' * 60}")
    print(f"City: {city}")
    print(f"Approximate tokens: {tokens}")
    print(f"Number of messages: {len(messages)}")
    print(f"{'=' * 60}")

    for message in messages:
        print(f"\n{type(message).__name__}:")
        print(message.content)


City: Paris
Approximate tokens: 46
Number of messages: 4

HumanMessage:
Find hotels in Paris

AIMessage:


ToolMessage:

Hotels in Paris:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $100/night, business center
3. Budget Stay - 3 star, $75/night, free wifi


AIMessage:


City: London
Approximate tokens: 93
Number of messages: 8

HumanMessage:
Find hotels in Paris

AIMessage:


ToolMessage:

Hotels in Paris:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $100/night, business center
3. Budget Stay - 3 star, $75/night, free wifi


AIMessage:


HumanMessage:
Find hotels in London

AIMessage:


ToolMessage:

Hotels in London:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $100/night, business center
3. Budget Stay - 3 star, $75/night, free wifi


AIMessage:


City: Tokyo
Approximate tokens: 140
Number of messages: 12

HumanMessage:
Find hotels in Paris

AIMessage:


ToolMessage:

Hotels in Paris:
1. Grand Ho

GoogleRateLimitError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 214.601781ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fre